In [2]:
import json
import re
import csv
from pathlib import Path

In [3]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

In [5]:
DATASET_FOLDER = Path("/Users/megan/Downloads/PMC_Patients")

In [6]:
# create path for output matrix
OUTPUT_FOLDER = DATASET_FOLDER / "tfidf_output"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

In [7]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to /Users/megan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/megan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/megan/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [8]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [9]:
def preprocess_text(text):
    # convert all text to lowercase so "Patients" is the same as "patient"
    text = text.lower()

    # remove characters that are not lowercase, numbers, or normal spaces
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    # tokenize terms
    words = text.split()

    # remove stop words (e.g., the, or, with) and lemmatization (patients and patient is same term)
    processed_words = []

    for word in words:
        if word not in stop_words:
            lemma = lemmatizer.lemmatize(word)
            processed_words.append(lemma)

    return " ".join(processed_words)

In [10]:
documents = []
document_ids = []

In [11]:
json_files = sorted(DATASET_FOLDER.rglob("*.json"))

In [12]:
print(f"Found {len(json_files)} JSON files.")

Found 8584 JSON files.


In [13]:
# I used the WordNet lemmatizer and now will extract the JSON documents
for file_path in json_files:

    if OUTPUT_FOLDER in file_path.parents:
        continue

    try:
        with open(file_path, "r", encoding="utf-8") as file:
            data = json.load(file)

        records = data if isinstance(data, list) else [data]

        for record in records:
            if not isinstance(record, dict):
                continue

            title = record.get("title", "")
            patient_text = record.get("patient", "")

            if not isinstance(title, str):
                title = ""

            if not isinstance(patient_text, str):
                patient_text = ""

            original_text = f"{title} {patient_text}".strip()

            
            if not original_text:
                continue

            processed_text = preprocess_text(original_text)

            documents.append(processed_text)

            # save identifiers for later
            document_ids.append({
                "filename": str(file_path.relative_to(DATASET_FOLDER)),
                "patient_id": record.get("patient_id", ""),
                "PMID": record.get("PMID", "")
            })

    except (json.JSONDecodeError, OSError) as error:
        print(f"Could not read {file_path}: {error}")


print(f"Successfully extracted {len(documents)} documents.")

Successfully extracted 8579 documents.


In [14]:
if not documents:
    raise ValueError(
        "No usable documents were found. "
        "Check the dataset folder and JSON structure."
    )

In [15]:
vectorizer = TfidfVectorizer(
    smooth_idf=True,
    sublinear_tf=True,
    norm="l2"
)

In [16]:
tfidf_matrix = vectorizer.fit_transform(documents)

print("TF-IDF matrix constructed successfully.")
print("Matrix dimensions:", tfidf_matrix.shape)

TF-IDF matrix constructed successfully.
Matrix dimensions: (8579, 61302)


In [28]:
feature_names = vectorizer.get_feature_names_out()
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=feature_names
)

tfidf_df.to_csv(
    OUTPUT_FOLDER / "tfidf_matrix2.csv",
    index=False
)

KeyboardInterrupt: 

In [17]:
# create a smaller matrix file
small_matrix = tfidf_matrix[:1000, :1000]

small_features = vectorizer.get_feature_names_out()[:1000]

small_df = pd.DataFrame(
    small_matrix.toarray(),
    columns=small_features
)

small_df.to_csv(
    OUTPUT_FOLDER / "tfidf_matrix_small.csv",
    index=False
)